# Chapter 4 — Structured Prompting
## AI-Based Data Engineering (Packt)

The **PTCF pattern** (Persona / Task / Context / Format) makes prompts version-controllable, diffable, testable, and composable.

| Benefit | How |
|---|---|
| **Diffable** | `git diff` shows exactly what changed between prompt versions |
| **Testable** | PromptFoo test cases (Section 4.4) evaluate against expected outputs |
| **Composable** | Same `DataEngineeringPrompt` class builds SQL, docs, and triage prompts |
| **API-enforced** | `tool_choice` + Pydantic guarantees valid JSON every time |

**What you'll build:** a PTCF-structured SQL generator that takes a business question and OpsPulse schema context and returns a validated `SQLGenerationResult`.

In [ ]:
import anthropic
import json
from dataclasses import dataclass, field
from typing import Optional
from pydantic import BaseModel, Field

client = anthropic.Anthropic()
print("Anthropic client ready.")

## The DataEngineeringPrompt Dataclass

The five constraint-set elements address three failure modes: model hallucination, dialect mismatch, and underspecified output.

| Element | Purpose |
|---|---|
| **Persona** | Scope, dialect, and authority constraints |
| **Task** | The business question, grounded in OpsPulse schema |
| **Context** | Structured JSON: schema, glossary, dialect notes |
| **Format** | Exact output contract — keys, types, cardinality |
| **Fallback** | What to return when context is insufficient |

In [ ]:
@dataclass
class DataEngineeringPrompt:
    """
    PTCF (Persona/Task/Context/Format) prompt builder.
    Encodes prompt components as a dataclass so they are version-controlled,
    diffable in git, and testable with PromptFoo (Section 4.4).
    """
    persona:              str
    task:                 str
    context:              dict
    output_format:        str
    fallback_instruction: str = (
        'If you cannot produce a valid answer with the provided context, return: '
        '{"error": "insufficient_context", "missing": "<describe what is missing>"}'
    )
    chain_of_thought: bool = False

    def to_system_message(self) -> str:
        parts = [self.persona, self.fallback_instruction]
        if self.chain_of_thought:
            parts.insert(1, 'Think step by step before producing your final answer.')
        return '\n\n'.join(parts)

    def to_user_message(self) -> str:
        return (
            f'Task: {self.task}\n\n'
            f'Context:\n{json.dumps(self.context, indent=2)}\n\n'
            f'Output format: {self.output_format}'
        )

    def to_messages(self) -> list[dict]:
        return [{'role': 'user', 'content': self.to_user_message()}]


print("DataEngineeringPrompt defined.")

In [ ]:
%%sql -r column_schema
-- Real schema context from OpsPulse marts layer
-- NOTE: requires OpsPulse generator (code/setup/opspulse_generator.py --target snowflake)
SELECT
    column_name,
    data_type,
    is_nullable,
    COALESCE(comment, '') AS comment
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position;

In [ ]:
# Build a schema dict from the SQL result
schema_rows = column_schema.to_pandas()
schema = {
    'table': 'OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS',
    'columns': [
        {
            'name':        row['COLUMN_NAME'],
            'type':        row['DATA_TYPE'],
            'nullable':    row['IS_NULLABLE'] == 'YES',
            'description': row['COMMENT'],
        }
        for _, row in schema_rows.iterrows()
    ],
}

# Build the PTCF prompt for SQL generation
prompt = DataEngineeringPrompt(
    persona=(
        'You are a senior Snowflake data engineer. '
        'You write read-only SELECT queries in Snowflake SQL dialect. '
        'You never reference tables not listed in the provided schema. '
        'You never use DML, DDL, or CALL statements.'
    ),
    task='How many active customers are in each region, ordered by count descending?',
    context={
        'schema':        schema,
        'dialect_notes': [
            'Use CURRENT_DATE, not NOW() or GETDATE()',
            "Date arithmetic: DATEADD('day', -7, CURRENT_DATE)",
            'Case-insensitive string compare: ILIKE, not LIKE',
            'Null-safe equality: IS NOT DISTINCT FROM, not =',
        ],
    },
    output_format=(
        'JSON with keys: '
        'sql (string — valid Snowflake SELECT), '
        'explanation (string — max 2 sentences), '
        'confidence (string — high | medium | low)'
    ),
    chain_of_thought=True,
)

# Call claude-haiku-4-5 for a quick, low-cost first pass
response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=600,
    system=prompt.to_system_message(),
    messages=prompt.to_messages(),
)
raw_response = response.content[0].text
print('System message (first 200 chars):')
print(prompt.to_system_message()[:200])
print('\nRaw response from claude-haiku-4-5:')
print(raw_response)

## API-Enforced Structured Output with `tool_choice`

The response above is unstructured text — valid JSON if the model cooperates, but fragile if it adds explanation prose around the JSON block.

**API enforcement** uses `tool_choice={"type": "tool", "name": "generate_sql"}` to force exactly one valid tool call that deserializes directly into a Pydantic model. This eliminates string parsing entirely.

The `SQLGenerationResult` model below is the output contract the API enforces.

In [ ]:
class SQLGenerationResult(BaseModel):
    sql:           str       = Field(description='Valid Snowflake SELECT statement')
    explanation:   str       = Field(description='What the query returns, max 2 sentences')
    confidence:    str       = Field(description='One of: high, medium, low')
    assumed_joins: list[str] = Field(default_factory=list, description='Inferred join conditions')


# Call with tool_choice to get API-enforced structured output
response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=600,
    system=prompt.to_system_message(),
    tools=[{
        'name': 'generate_sql',
        'description': 'Generate a Snowflake SQL query answering the business question.',
        'input_schema': SQLGenerationResult.model_json_schema(),
    }],
    tool_choice={'type': 'tool', 'name': 'generate_sql'},
    messages=prompt.to_messages(),
)

tool_call = next(b for b in response.content if b.type == 'tool_use')
result    = SQLGenerationResult(**tool_call.input)

print(f'SQL:\n{result.sql}\n')
print(f'Explanation: {result.explanation}')
print(f'Confidence:  {result.confidence}')
if result.assumed_joins:
    print(f'Assumed joins: {result.assumed_joins}')

In [ ]:
%%sql -r query_result
-- Execute the query the PTCF + tool_choice pipeline generated
-- NOTE: requires OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS from the OpsPulse generator
SELECT
    region_code,
    COUNT(*) AS active_count
FROM OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS
GROUP BY region_code
ORDER BY active_count DESC;

## Summary

| Technique | Benefit | Trade-off |
|---|---|---|
| **PTCF dataclass** | Version-controlled, testable prompts | More setup than an inline string |
| **Chain-of-thought** | Better SQL for multi-join queries | +100–200 tokens per call |
| **`tool_choice` enforcement** | Guaranteed valid JSON; no string parsing | Requires tool-use support |

The PTCF pattern is composable: the same `DataEngineeringPrompt` class builds the column documentation prompt in Chapter 8 and the triage prompt in Chapter 3. See `code/ch04_structured_prompting/data_engineering_prompt.py` for the full implementation including PromptFoo test cases.